# K-Means Clustering, Explained Simply
Finding groups in data with no labels at all. Runs free in Google Colab.

Run each cell from top to bottom with **Shift + Enter**.

## 1. The data
A shop knows two things about 300 customers: how much they spend in a year, and how many times a month they visit. There are no labels: nobody has told us any groups.

### 300 customers
- `customers.csv`: 300 customers: yearly spend in dollars, and visits per month

In [ ]:
import pandas as pd

url = ("https://raw.githubusercontent.com/Uttam-Nepal/"
       "machine-learning-explained/main/"
       "k-means-clustering/customers.csv")
df = pd.read_csv(url)
print(df.shape)
df.head()

### Look at the customers
- `the picture`: the eye spots a few clumps, but nothing tells the computer where they are

In [ ]:
import matplotlib.pyplot as plt

plt.scatter(df["visits_per_month"], df["yearly_spend"], s=15)
plt.xlabel("visits per month"); plt.ylabel("yearly spend ($)")
plt.show()

## 2. K-means
Pick k centres. Then repeat: (1) give every customer to its nearest centre, (2) move each centre to the average of its customers. Stop when nothing changes. Scale the inputs first, because k-means measures distances.

### Find 4 groups
- `StandardScaler`: puts dollars and visits on the same scale
- `n_clusters=4`: k = 4 groups
- `n_init=10`: run 10 times from different starting centres, keep the best
- `km.labels_`: the group number (0 to 3) given to each customer

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

scaler = StandardScaler()
Z = scaler.fit_transform(df)
km = KMeans(n_clusters=4, n_init=10, random_state=0).fit(Z)
df["group"] = km.labels_
print(df["group"].value_counts().sort_index().to_dict())

### The groups and their centres
- `inverse_transform`: turn the centres back into dollars and visits
- `marker="X"`: each group's centre as a big black X

In [ ]:
centres = scaler.inverse_transform(km.cluster_centers_)
plt.scatter(df["visits_per_month"], df["yearly_spend"],
            c=df["group"], cmap="tab10", s=15)
plt.scatter(centres[:, 1], centres[:, 0], c="black", marker="X", s=200)
plt.xlabel("visits per month"); plt.ylabel("yearly spend ($)")
plt.show()

### Watch k-means work, step by step
- `rng.choice(..., 4)`: start with 4 random customers as the centres
- `dist`: the squared distance from every customer to every centre
- `dist.argmin(axis=1)`: each customer joins its nearest centre
- `Z[label == k].mean(axis=0)`: move each centre to the average of its customers

In [ ]:
import numpy as np

rng = np.random.default_rng(3)
centre = Z[rng.choice(len(Z), 4, replace=False)]
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for step, ax in enumerate(axes):
    dist = ((Z[:, None, :] - centre[None, :, :]) ** 2).sum(axis=2)
    label = dist.argmin(axis=1)
    ax.scatter(Z[:, 1], Z[:, 0], c=label, cmap="tab10", s=8)
    ax.scatter(centre[:, 1], centre[:, 0], c="black", marker="X", s=150)
    ax.set_title(f"step {step + 1}")
    centre = np.array([Z[label == k].mean(axis=0) for k in range(4)])
plt.show()

## 3. Choosing k
**Inertia**: the total squared distance from each customer to its centre. It always falls as k grows, so look for the **elbow**, where it stops falling fast.

### The elbow method
- `inertia_`: the sum of squared distances to the nearest centre
- `the elbow`: big drops up to k = 4, then only small ones

In [ ]:
ks = range(1, 9)
inertia = [KMeans(n_clusters=k, n_init=10, random_state=0).fit(Z).inertia_
           for k in ks]
plt.plot(ks, inertia, marker="o")
plt.xlabel("k (number of groups)"); plt.ylabel("inertia")
plt.show()
print([round(i) for i in inertia])

## 4. Describe the groups
Clustering gives numbers, not meanings. We look at each group and name it ourselves.

### What is each group like?
- `groupby("group")`: one row per group
- `agg(...)`: count the customers, and average the spend and visits

In [ ]:
profile = df.groupby("group").agg(customers=("group", "size"),
                                  spend=("yearly_spend", "mean"),
                                  visits=("visits_per_month", "mean")).round(1)
print(profile)

### Which group is a new customer in?
- `scaler.transform(new)`: scale the new customers the same way
- `km.predict`: the nearest centre for each one

In [ ]:
new = pd.DataFrame({"yearly_spend": [300, 3500], "visits_per_month": [1, 7]})
print(km.predict(scaler.transform(new)))

## Try it yourself
1. Which group would a customer spending $2,500 a year with 3 visits a month join?
2. Run KMeans with k = 3 and k = 6. Do the groups still make sense?
3. Fit KMeans without scaling. How do the groups change?